In [1]:
import requests
import os
import json
from dotenv import load_dotenv
import pandas as pd

# Carrega o .env
load_dotenv()
app_key = os.getenv("app_key")
app_secret = os.getenv("app_secret")

# API endpoint
url = "https://app.omie.com.br/api/v1/geral/clientes/"

# Parameters for the API call
param = {
    "pagina": 1,
    "registros_por_pagina": 50,  # Adjust as needed
    "apenas_importado_api": "N"
}

all_clients = []

while True:
    payload = {
        "call": "ListarClientes",
        "app_key": app_key,
        "app_secret": app_secret,
        "param": [param]
    }
    
    response = requests.post(url, json=payload)
    
    if response.status_code != 200:
        print(f"Error: {response.status_code}")
        break
    
    data = response.json()
    
    if "clientes_cadastro" in data:
        all_clients.extend(data["clientes_cadastro"])
    
    # Check if there are more pages
    total_pages = data.get("total_de_paginas", 1)
    current_page = data.get("pagina", 1)
    
    if current_page >= total_pages:
        break
    
    param["pagina"] += 1

# Now all_clients contains all the client data
print(f"Total clients retrieved: {len(all_clients)}")


# Extract CNPJ and nome fantasia from each client
extracted_data = []
for client in all_clients:
    cnpj = client.get('cnpj_cpf', '')
    nome_fantasia = client.get('nome_fantasia', '')
    extracted_data.append({'CNPJ': cnpj, 'Nome Fantasia': nome_fantasia})

# Create a DataFrame
df = pd.DataFrame(extracted_data)

# Export to CSV
df.to_csv('clientes_extracao_distribuidora.csv', index=False, encoding='utf-8-sig')

print("Dados extraídos e exportados para 'clientes_extracao_distribuidora.csv'")
print(f"Total de clientes processados: {len(extracted_data)}")

Total clients retrieved: 255
Dados extraídos e exportados para 'clientes_extracao_distribuidora.csv'
Total de clientes processados: 255
